# Stream E - Packaging & scale-out (M5)

**Milestone M5:** one full patient processed end-to-end by a single command, with
measured time/storage and full provenance - the basis for extrapolating to 3,000.

Requires these next to this notebook:
`run_pipeline.py`, `measure.py`, `provenance.py`, `cutter_wsidicom.py`,
`tiler_wsidicom.py`, `stain_wsidicom.py`.

| task | what it delivers |
|---|---|
| E1 | one CLI, every constant exposed as a parameter |
| E2 | tiles-in / tiles-surviving / wall-clock / disk + x3,000 extrapolation |
| E3 | manifest tying each tile to patient+slide+coords; patient-level split |

## Setup

In [3]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os, sys, subprocess
import numpy as np
import pandas as pd

import measure as M
import provenance as P

SLIDE = ("/Users/yahyaamjad/Downloads/Research/cptac_brca/01BR001/"
         "2.25.48791557373299768401597362411459861639/"
         "SM_1.3.6.1.4.1.5962.99.1.132039251.338821108.1640809579091.2.0")   # the FOLDER
OUT     = "./out/01BR001"
PATIENT = "01BR001"
SLIDE_ID = os.path.basename(os.path.normpath(SLIDE))
REFERENCE = "colorstandard.png"    # from SD.save_reference(); None to skip Stream D
os.makedirs(OUT, exist_ok=True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


---
## E1 - One command, parameters exposed

Wraps B+C+D into a single CLI. Panoptes' orchestration (`prep.cutter()` +
`execute.py`) but parameterized: tile size, thresholds, reference image and
resolution are arguments, not constants.

**Teach deliverable** - every parameter and why it is exposed. `--help` prints it:

In [ ]:
!python run_pipeline.py --help

### Run it on one slide

`--serial` first on a new slide (real tracebacks); switch to `--parallel` once clean.
Drop `--reference` to skip Stream D normalization.

In [4]:
cmd = [sys.executable, "run_pipeline.py",
       "--slide", SLIDE,
       "--out", OUT,
       "--patient", PATIENT,
       "--slide-id", SLIDE_ID,
       "--targets", "2", "4", "8",
       "--blank-max", "0.8",
       "--edge", "pad",
       "--serial",
       "--cohort-n", "3000"]
if REFERENCE and os.path.exists(REFERENCE):
    cmd += ["--reference", REFERENCE, "--norm-method", "macenko"]

print(" ".join(cmd), "\n")
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout[-6000:])
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr[-3000:])

/Users/yahyaamjad/Downloads/Research/wsi-processing-pipeline/.venv/bin/python run_pipeline.py --slide /Users/yahyaamjad/Downloads/Research/cptac_brca/01BR001/2.25.48791557373299768401597362411459861639/SM_1.3.6.1.4.1.5962.99.1.132039251.338821108.1640809579091.2.0 --out ./out/01BR001 --patient 01BR001 --slide-id SM_1.3.6.1.4.1.5962.99.1.132039251.338821108.1640809579091.2.0 --targets 2 4 8 --blank-max 0.8 --edge pad --serial --cohort-n 3000 --reference colorstandard.png --norm-method macenko 

== E1 pipeline ==
slide   : /Users/yahyaamjad/Downloads/Research/cptac_brca/01BR001/2.25.48791557373299768401597362411459861639/SM_1.3.6.1.4.1.5962.99.1.132039251.338821108.1640809579091.2.0
patient : 01BR001 | slide id: SM_1.3.6.1.4.1.5962.99.1.132039251.338821108.1640809579091.2.0
targets : [2, 4, 8] | blank_max: 0.8 | edge: pad | parallel: False
available downsamples (list index -> factor): {0: 1, 1: 4, 2: 16}
ladder: level1=(list 0, ft 2, eff 2x), level2=(list 1, ft 1, eff 4x), level3=(list 1

---
## E2 - One patient end-to-end + measurement (M5)

The four numbers, then extrapolate x3,000 and name the bottleneck.
The CLI already printed this; below re-derives it in-notebook so you can vary the
cohort size, storage budget and deadline without re-tiling.

In [5]:
meas = pd.read_csv(os.path.join(OUT, "measurements.csv"), index_col=0).iloc[:, 0]
meas

tiles_in             6.422000e+03
tiles_surviving      6.600000e+02
keep_rate            1.027717e-01
wall_clock_s         3.845097e+01
disk_bytes           1.108453e+08
per_tile_ms          5.825905e+01
bytes_per_tile       1.679475e+05
workers              8.000000e+00
cohort_n             3.000000e+03
cohort_time_s        1.153529e+05
cohort_disk_bytes    3.325360e+11
cohort_tiles         1.980000e+06
Name: 0, dtype: float64

### Re-run the extrapolation against your real budget/deadline

In [6]:
# rebuild the report dict from the saved measurements
results = {}
for m in (1, 2, 3):
    d = os.path.join(OUT, "level{}".format(m))
    csv = os.path.join(d, "dict.csv")
    if os.path.exists(csv):
        n = len(pd.read_csv(csv))
        results[m] = {"n_x": 0, "n_y": 0, "count": n}

rep = M.measure_run(OUT, results, float(meas["wall_clock_s"]),
                    n_patients=3000, workers=int(meas["workers"]))
rep["tiles_in"] = int(meas["tiles_in"])          # keep the true candidate count
rep["keep_rate"] = rep["tiles_surviving"] / rep["tiles_in"]

M.print_report(rep,
               storage_budget_tb=5.0,     # <- your actual budget
               deadline_days=14)          # <- your actual deadline
None

E2 - ONE PATIENT END-TO-END
tiles considered (in) : 6,422
tiles surviving (out) : 660  (10.3% kept)
wall clock            : 38.5 s  (8 workers)
disk used             : 105.8 MB
  per surviving tile  : 58.3 ms, 164.2 KB

per level:
  level1: grid 0x0 = 0 candidates -> 485 kept
  level2: grid 0x0 = 0 candidates -> 139 kept
  level3: grid 0x0 = 0 candidates -> 36 kept

EXTRAPOLATION x3,000 PATIENTS  (linear, 1 slide/patient)
total tiles  : 1,980,000
total time   : 1.3 days   (serial on this machine)
total storage: 310.0 GB

wall-clock if run across N machines like this one:
     1 machines: 1.3 days
     4 machines: 8.0 h
    10 machines: 3.2 h
    50 machines: 38.5 min

bottleneck:
  - 0.3 TB / 1.3 days serial - compare against your actual budget and deadline
  - dominant cost is 164.2 KB per tile at 10% keep rate; the cheapest lever is the blank threshold (A4) and dropping a resolution (A1)


### Sensitivity: what changes the cohort cost?

Panoptes baselines these measurements test: per-slide `mp.Pool(cpu_count())` in
`Slicer.tile()`, and a set-size cap of `batchsize*80000/3` in `set_sep()`.
The two cheapest levers are the blank threshold (**A4**) and dropping a
resolution (**A1**) - both change tile count roughly linearly.

In [7]:
rows = []
for label, factor in [("as measured", 1.00),
                      ("blank_max stricter (-20% tiles)", 0.80),
                      ("drop level3 (-~5% tiles)", 0.95),
                      ("drop to 2 levels (-~33% tiles)", 0.67)]:
    rows.append({
        "scenario": label,
        "tiles / patient": int(rep["tiles_surviving"] * factor),
        "cohort tiles": int(rep["cohort_tiles"] * factor),
        "cohort storage": M.human_bytes(rep["cohort_disk_bytes"] * factor),
        "cohort time (serial)": M.human_time(rep["cohort_time_s"] * factor),
    })
pd.DataFrame(rows)

,scenario,tiles / patient,cohort tiles,cohort storage,cohort time (serial)
0,as measured,660,1980000,310.0 GB,1.3 days
1,blank_max stricter (-20% tiles),528,1584000,248.0 GB,1.1 days
2,drop level3 (-~5% tiles),627,1881000,294.5 GB,1.3 days
3,drop to 2 levels (-~33% tiles),442,1326600,207.7 GB,21.5 h


### M5 write-up

Fill in from the numbers above:

- **tiles considered / surviving:** ____ / ____ (____ % kept)
- **wall clock (one patient):** ____ on ____ workers
- **disk (one patient):** ____
- **x3,000 projection:** ____ tiles, ____ storage, ____ serial time
- **bottleneck:** ____ (storage vs compute vs I/O), because ____
- **mitigation:** ____ (parallelise across N machines / raise blank_max per A4 /
  drop a resolution per A1)

---
## E3 - Provenance & patient-level split readiness

Every tile must trace to patient + slide + slide-coordinates, and splits must be
by patient. Panoptes reference: `sample_prep.set_sep()` splits on
`subset.slide.unique()`; the dict CSVs (`Num,X_pos,Y_pos,X,Y,Loc`) are the
provenance record. Both reproduced.

### The manifest

In [8]:
manifest = pd.read_csv(os.path.join(OUT, "manifest.csv"))
print(manifest.shape, "|", manifest.patient.nunique(), "patient(s),",
      manifest.level.nunique(), "levels")
manifest.head()

(660, 10) | 1 patient(s), 3 levels


,patient,slide,level,Num,X_pos,Y_pos,X,Y,Loc,eff_level_dir
0,01BR001,SM_1.3.6.1.4.1.5962.99.1.132039251.338821108.1...,1,0,8,26,4000,13000,./out/01BR001/level1/region_x-4000-y-13000.png,./out/01BR001/level1
1,01BR001,SM_1.3.6.1.4.1.5962.99.1.132039251.338821108.1...,1,1,8,27,4000,13500,./out/01BR001/level1/region_x-4000-y-13500.png,./out/01BR001/level1
2,01BR001,SM_1.3.6.1.4.1.5962.99.1.132039251.338821108.1...,1,2,8,28,4000,14000,./out/01BR001/level1/region_x-4000-y-14000.png,./out/01BR001/level1
3,01BR001,SM_1.3.6.1.4.1.5962.99.1.132039251.338821108.1...,1,3,8,29,4000,14500,./out/01BR001/level1/region_x-4000-y-14500.png,./out/01BR001/level1
4,01BR001,SM_1.3.6.1.4.1.5962.99.1.132039251.338821108.1...,1,4,8,30,4000,15000,./out/01BR001/level1/region_x-4000-y-15000.png,./out/01BR001/level1


In [9]:
P.check_provenance(manifest, verify_files=True)

{'ok': True,
 'tiles': 660,
 'patients': 1,
 'slides': 1,
 'levels': [1, 2, 3],
 'issues': ['none']}

### Build a cohort manifest

One patient cannot demonstrate a split. Point `PATIENTS` at each processed slide
folder (re-run E1 per patient), then merge.

In [10]:
PATIENTS = {
    # "01BR001": "./out/01BR001",
    # "01BR002": "./out/01BR002",
}

if PATIENTS:
    frames = [P.build_manifest(d, pid, pid, out_path=None)
              for pid, d in PATIENTS.items()]
    cohort = P.merge_manifests(frames, out_path="./out/cohort_manifest.csv")
else:
    cohort = manifest.copy()   # single-patient fallback so the cells below run
print(len(cohort), "tiles |", cohort.patient.nunique(), "patients")

660 tiles | 1 patients


### Patient-level split (the stub)

In [11]:
split = P.set_sep(cohort, cut=0.2, seed=0)     # asserts no patient spans splits
split.to_csv("./out/cohort_manifest_split.csv", index=False)
split.groupby(["split", "patient"]).size().head(20)

split by patient (cut=0.2): 1 train / 0 val / 0 test patients
tiles: {'train': 660}


split  patient
train  01BR001    660
dtype: int64

### Teach deliverable - why a by-tile split inflates metrics

`set_sep()` splits on the **patient**, never the tile. Demonstrate what the
alternative does: assign tiles to splits at random and count how many patients
end up in more than one split.

In [12]:
rng = np.random.default_rng(0)
by_tile = cohort.copy()
by_tile["split"] = rng.choice(["train", "val", "test"], size=len(cohort), p=[.6, .2, .2])

spanning = (by_tile.groupby("patient")["split"].nunique() > 1).sum()
print("BY-TILE split : {} of {} patients appear in >1 split".format(
    spanning, cohort.patient.nunique()))

spanning_ok = (split.groupby("patient")["split"].nunique() > 1).sum()
print("BY-PATIENT    : {} of {} patients appear in >1 split".format(
    spanning_ok, cohort.patient.nunique()))

BY-TILE split : 1 of 1 patients appear in >1 split
BY-PATIENT    : 0 of 1 patients appear in >1 split


**The argument.** Tiles from one slide are highly correlated: same patient, same
stain batch, same scanner, and consecutive tiles physically overlap by 49px. A
by-tile split therefore puts near-duplicate tiles in train *and* test. The model
can score well by recognising *this patient's* appearance rather than the
biology - so test accuracy measures memorisation, not generalisation, and the
inflation is invisible because the test set looks large and independent.

`set_sep()` prevents it structurally: the split unit is
`subset.slide.unique()`, so every tile from a patient lands in exactly one split.
The assertion inside `P.set_sep` enforces that invariant rather than trusting it.